In [0]:
SELECT 
 DISTINCT PRODUCT_NAME
 ,PRODUCTGROUP_NAME
FROM pricing_analytics.silver.daily_pricing_silver
WHERE PRODUCT_NAME='Onion'


PRODUCT_NAME,PRODUCTGROUP_NAME
Onion,Oil Seeds


In [0]:
create table if not exists pricing_analytics.gold.reporting_dim_product_gold_SCDTYPE1
like pricing_analytics.gold.reporting_dim_product_gold

In [0]:
SELECT * FROM pricing_analytics.gold.reporting_dim_product_gold_SCDTYPE1
WHERE PRODUCT_NAME='Onion'

PRODUCTGROUP_NAME,PRODUCT_NAME,PRODUCT_ID,lakehouse_inserted_date,lakehouse_updated_date
Vegetables,Onion,147,2026-02-23T20:40:40.972Z,2026-02-23T21:09:32.189Z


In [0]:
UPDATE pricing_analytics.silver.daily_pricing_silver
SET PRODUCTGROUP_NAME='Oil Seeds',
lakehouse_update_date = current_timestamp()
WHERE PRODUCT_NAME='Onion'

num_affected_rows
547


In [0]:
CREATE OR REPLACE TABLE pricing_analytics.silver.reporting_dim_product_stage_1 AS
SELECT 
 DISTINCT PRODUCT_NAME
 ,PRODUCTGROUP_NAME
FROM pricing_analytics.silver.daily_pricing_silver
WHERE lakehouse_update_date > (SELECT nvl(max(PROCESSED_TABLE_DATETIME),'1900-01-01') FROM pricing_analytics.processrunlogs.DELTALAKEHOUSE_PROCESS_RUNS 
WHERE process_name = 'reportingDimensionTablesLoadScdType1' AND process_status = 'Completed' );

num_affected_rows,num_inserted_rows


In [0]:
select * from pricing_analytics.silver.reporting_dim_product_stage_1

PRODUCT_NAME,PRODUCTGROUP_NAME
Onion,Oil Seeds


In [0]:
CREATE OR REPLACE TABLE pricing_analytics.silver.reporting_dim_product_stage_2 AS 
SELECT 
  silverDim.PRODUCT_NAME
  ,silverDim.PRODUCTGROUP_NAME,
  golddim.PRODUCT_NAME as GOLD_PRODUCT_NAME
 ,case when goldDim.PRODUCT_NAME IS NULL 
 then ROW_NUMBER() OVER (  ORDER BY silverDim.PRODUCT_NAME,silverDim.PRODUCTGROUP_NAME)  
 else goldDim.PRODUCT_ID end as PRODUCT_ID
 ,current_timestamp() as lakehouse_inserted_date
 ,current_timestamp() as lakehouse_updated_date
FROM pricing_analytics.silver.reporting_dim_product_stage_1 silverDim
LEFT OUTER JOIN pricing_analytics.gold.reporting_dim_product_gold_scdtype1 goldDim
ON silverDim.PRODUCT_NAME= goldDim.PRODUCT_NAME
WHERE goldDim.PRODUCT_NAME IS NULL or silverdim.PRODUCTGROUP_NAME <> goldDim.PRODUCTGROUP_NAME;

num_affected_rows,num_inserted_rows


In [0]:
select * from pricing_analytics.silver.reporting_dim_product_stage_2

PRODUCT_NAME,PRODUCTGROUP_NAME,GOLD_PRODUCT_NAME,PRODUCT_ID,lakehouse_inserted_date,lakehouse_updated_date
Onion,Oil Seeds,Onion,147,2026-02-23T21:30:49.537Z,2026-02-23T21:30:49.537Z


In [0]:
CREATE OR REPLACE TABLE pricing_analytics.silver.reporting_dim_product_stage_3 AS 
SELECT
  silverDim.PRODUCTGROUP_NAME
  ,silverDim.PRODUCT_NAME
,case when GOLD_PRODUCT_NAME IS NULL 
then silverDim.PRODUCT_ID + PREV_MAX_SK_ID 
else PRODUCT_ID end as PRODUCT_ID
,current_timestamp() as lakehouse_inserted_date
,current_timestamp() as lakehouse_updated_date
FROM 
pricing_analytics.silver.reporting_dim_product_stage_2 silverDim
CROSS JOIN (SELECT nvl(MAX(PRODUCT_ID),0) as PREV_MAX_SK_ID FROM pricing_analytics.gold.reporting_dim_product_gold_SCDTYPE1 ) goldDim;

num_affected_rows,num_inserted_rows


In [0]:
select * from pricing_analytics.silver.reporting_dim_product_stage_3

PRODUCTGROUP_NAME,PRODUCT_NAME,PRODUCT_ID,lakehouse_inserted_date,lakehouse_updated_date
Vegetables,Onion,216,2026-02-23T21:09:16.247Z,2026-02-23T21:09:16.247Z


In [0]:
merge into pricing_analytics.gold.reporting_dim_product_gold_SCDTYPE1 goldDim
using pricing_analytics.silver.reporting_dim_product_stage_3 silverDim
on goldDim.PRODUCT_NAME = silverDim.PRODUCT_NAME
when matched then 
update set goldDim.PRODUCTGROUP_NAME = silverDim.PRODUCTGROUP_NAME,
goldDim.lakehouse_updated_date = current_timestamp()
when not matched then 
insert *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,1,0,0


In [0]:
SELECT * FROM pricing_analytics.gold.reporting_dim_product_gold_SCDTYPE1
WHERE PRODUCT_NAME='Onion'

PRODUCTGROUP_NAME,PRODUCT_NAME,PRODUCT_ID,lakehouse_inserted_date,lakehouse_updated_date
Oil Seeds,Onion,147,2026-02-23T20:40:40.972Z,2026-02-23T21:37:15.136Z


In [0]:
INSERT INTO  pricing_analytics.processrunlogs.DELTALAKEHOUSE_PROCESS_RUNS(PROCESS_NAME,PROCESSED_TABLE_DATETIME,PROCESS_STATUS)
SELECT 'reportingDimensionTablesLoadScdType1' , max(lakehouse_update_date) ,'Completed' FROM pricing_analytics.silver.daily_pricing_silver

num_affected_rows,num_inserted_rows
1,1
